# 文本分类


In [1]:
import pandas as pd

from transformers import AutoTokenizer

data = pd.read_csv('ChnSentiCorp_htl_all.csv')
data


D:\PycharmProjects\llm_sft\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,label,review
0,1,"距离川沙公路较近,但是公交指示不对,如果是""蔡陆线""的话,会非常麻烦.建议用别的路线.房间较..."
1,1,商务大床房，房间很大，床有2M宽，整体感觉经济实惠不错!
2,1,早餐太差，无论去多少人，那边也不加食品的。酒店应该重视一下这个问题了。房间本身很好。
3,1,宾馆在小街道上，不大好找，但还好北京热心同胞很多~宾馆设施跟介绍的差不多，房间很小，确实挺小...
4,1,"CBD中心,周围没什么店铺,说5星有点勉强.不知道为什么卫生间没有电吹风"
...,...,...
7761,0,尼斯酒店的几大特点：噪音大、环境差、配置低、服务效率低。如：1、隔壁歌厅的声音闹至午夜3点许...
7762,0,盐城来了很多次，第一次住盐阜宾馆，我的确很失望整个墙壁黑咕隆咚的，好像被烟熏过一样家具非常的...
7763,0,看照片觉得还挺不错的，又是4星级的，但入住以后除了后悔没有别的，房间挺大但空空的，早餐是有但...
7764,0,我们去盐城的时候那里的最低气温只有4度，晚上冷得要死，居然还不开空调，投诉到酒店客房部，得到...


In [2]:
data = data.dropna()
data

,label,review
0,1,"距离川沙公路较近,但是公交指示不对,如果是""蔡陆线""的话,会非常麻烦.建议用别的路线.房间较..."
1,1,商务大床房，房间很大，床有2M宽，整体感觉经济实惠不错!
2,1,早餐太差，无论去多少人，那边也不加食品的。酒店应该重视一下这个问题了。房间本身很好。
3,1,宾馆在小街道上，不大好找，但还好北京热心同胞很多~宾馆设施跟介绍的差不多，房间很小，确实挺小...
4,1,"CBD中心,周围没什么店铺,说5星有点勉强.不知道为什么卫生间没有电吹风"
...,...,...
7761,0,尼斯酒店的几大特点：噪音大、环境差、配置低、服务效率低。如：1、隔壁歌厅的声音闹至午夜3点许...
7762,0,盐城来了很多次，第一次住盐阜宾馆，我的确很失望整个墙壁黑咕隆咚的，好像被烟熏过一样家具非常的...
7763,0,看照片觉得还挺不错的，又是4星级的，但入住以后除了后悔没有别的，房间挺大但空空的，早餐是有但...
7764,0,我们去盐城的时候那里的最低气温只有4度，晚上冷得要死，居然还不开空调，投诉到酒店客房部，得到...


## 创建Dataset

In [3]:
from torch.utils.data import Dataset


class MyDataset(Dataset):
    def __init__(self):
        super().__init__()
        self.data = pd.read_csv('ChnSentiCorp_htl_all.csv')
        self.data = self.data.dropna()

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        return self.data.iloc[index]['review'], self.data.iloc[index]['label']


In [4]:
dataset = MyDataset()

for i in range(5):
    print(dataset[i])

('距离川沙公路较近,但是公交指示不对,如果是"蔡陆线"的话,会非常麻烦.建议用别的路线.房间较为简单.', 1)
('商务大床房，房间很大，床有2M宽，整体感觉经济实惠不错!', 1)
('早餐太差，无论去多少人，那边也不加食品的。酒店应该重视一下这个问题了。房间本身很好。', 1)
('宾馆在小街道上，不大好找，但还好北京热心同胞很多~宾馆设施跟介绍的差不多，房间很小，确实挺小，但加上低价位因素，还是无超所值的；环境不错，就在小胡同内，安静整洁，暖气好足-_-||。。。呵还有一大优势就是从宾馆出发，步行不到十分钟就可以到梅兰芳故居等等，京味小胡同，北海距离好近呢。总之，不错。推荐给节约消费的自助游朋友~比较划算，附近特色小吃很多~', 1)
('CBD中心,周围没什么店铺,说5星有点勉强.不知道为什么卫生间没有电吹风', 1)


In [5]:
from torch.utils.data import random_split

train_set, valid_set = random_split(dataset, [0.9, 0.1])
len(train_set), len(valid_set)

(6989, 776)

In [6]:
for i in range(10):
    print(train_set[i])

('去的时候已经很晚了，在大厅叫了好几声才有一个服务员从前台后面爬起来，帮忙办理了入住手续。进到房间后发现房间的设施非常简陋，霉味也非常大。第二天早上起来才仔细看了一下房间，比起如家等快捷酒店差远了，说是三星的酒店，但距离三星不是差一点点。。。根本对不起220的价格，最多也就值100多一点点。', 0)
('这是我在携程遇到过的最差的酒店，问题多多1.这大冷天的。房间塑钢窗密封性极差，四处漏风，第二天换了个房间，也漏，不过比上一间漏的小点，联系服务员，服务员跟我说你自己拿透明胶条把缝粘粘就行2.暖气不热。服务员告知本酒店采用地暖方式，我一点没体会到地暖的优越性，在房间呆着需要靠羽绒服保暖，晚上睡觉需要盖2床被子3.浴室里淋浴的水时冷时热。水及不稳定，又时候冰你一下，有时候烫你一下4.房间里的空调禁止使用。从一进屋我就开始找空调遥控器，没找到，遂联系服务员，被告知由于我店采用地暖，空调暖风可以不用，遥控器由酒店统一保管。几经交涉终于把遥控器要到手了。。想要点暖风真不容易啊5.饭菜质量差。18一位的自助午餐，要了一份猪蹄，刚吃几口发现猪蹄上还有好多没拔干净的毛。。。。。宾馆反馈2008年6月10日：1、关于用水：经过我们认真的检查，我们最终确定造成洗浴水温波动的主要原因是冷水的压力变动。邯郸市自来水管网中的水压在用水峰谷压力不稳定，酒店专用自来水入户表井口径是50的，用水量不够使用。我们采取了以下措施：1、向邯郸市自来水公司申请扩容，安装直径100的入户表井，保证用水量，目前已完成。在管网中增加自动管道加压泵，保证恒定的管网压力，目前已完成。3、增加管网末端稳压水箱，体积3.3立方。经过以上措施彻底保证管网压力稳定。2、关于窗体密封：关于部分双层隔音窗密封问题，我酒店积极和施工单位进行协调，采取方法加以改善：更换部分窗扇的密封条。修复损坏失效的五金件。对窗体和楼体的接触部分，从内部进行密封胶密封，从外部进行玻璃胶密封。以上措施，业已落实到位，效果明显。3、关于地采暖：对于燕赵之星开业以来的第一个冬季，我们在取暖工作上的确出现了一些问题。首先是，我们对采暖系统的操作上经验不足，设备运能不能完全发挥，对此，我们积极联系厂家和专业维修人员进行设备调试。另外我们对具体操作的员工进行了调整和培训，制定完善了操作规程，明确了岗位责任。同时，我们对原煤供货商进行了调整，采购热能更高，

In [24]:
import torch

tokenizer = AutoTokenizer.from_pretrained('hfl/rbt3')


def collate_fn(batch):
    texts = [i[0] for i in batch]
    labels = [i[1] for i in batch]

    inputs = tokenizer(texts, max_length=128, padding="max_length", truncation=True, return_tensors="pt")
    # print(inputs)
    inputs['labels'] = torch.tensor(labels)

    return inputs

In [25]:
from torch.utils.data import DataLoader

trainloader = DataLoader(train_set, batch_size=32, shuffle=True, collate_fn=collate_fn)
validloader = DataLoader(valid_set, batch_size=64, shuffle=True, collate_fn=collate_fn)

In [26]:
next(enumerate(trainloader))[1]

{'input_ids': tensor([[ 101, 1283,  674,  ...,    0,    0,    0],
        [ 101,  679, 7231,  ...,    0,    0,    0],
        [ 101, 1041, 1071,  ...,    0,    0,    0],
        ...,
        [ 101, 6983, 2421,  ...,    0,    0,    0],
        [ 101, 2769, 3221,  ..., 2792,  809,  102],
        [ 101, 2769,  738,  ..., 2523, 3191,  102]]), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1]]), 'labels': tensor([0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0,
        1, 1, 0, 1, 1, 1, 1, 1])}

## 创建模型


In [27]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained('hfl/rbt3')
if torch.cuda.is_available():
    model = model.cuda()

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at hfl/rbt3 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [28]:

from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=2e-5)

In [29]:
def evaluate():
    model.eval()
    acc_num = 0
    with torch.inference_mode():
        for batch in validloader:
            if torch.cuda.is_available():
                batch = {k: v.cuda() for k, v in batch.items()}
            output = model(**batch)
            pred = torch.argmax(output.logits, dim=-1)
            acc_num += (pred.long() == batch['labels'].long()).float().sum()
    return acc_num / len(validloader.dataset)


def train(epoch=3, log_step=100):
    global_step = 0
    for ep in range(epoch):
        model.train()
        for batch in trainloader:
            if torch.cuda.is_available():
                batch = {k: v.cuda() for k, v in batch.items()}
            optimizer.zero_grad()
            output = model(**batch)
            output.loss.backward()
            optimizer.step()
            if global_step % log_step == 0:
                print(f"ep: {ep}, global_step: {global_step}, loss: {output.loss.item()}")
            global_step += 1
        acc = evaluate()
        print(f"ep: {ep}, global_step: {global_step}, acc: {acc}")


In [30]:
train()

ep: 0, global_step: 0, loss: 0.6275644898414612
ep: 0, global_step: 100, loss: 0.26984158158302307
ep: 0, global_step: 200, loss: 0.22550217807292938
ep: 0, global_step: 219, acc: 0.8969072103500366
ep: 1, global_step: 300, loss: 0.22268180549144745
ep: 1, global_step: 400, loss: 0.21390141546726227
ep: 1, global_step: 438, acc: 0.9046391248703003
ep: 2, global_step: 500, loss: 0.07370308041572571
ep: 2, global_step: 600, loss: 0.2798287868499756
ep: 2, global_step: 657, acc: 0.9046391248703003


In [34]:
sen = "我很喜欢这个商品"
sen = "我不想再购买这个商品了"
model.eval()
with torch.inference_mode():
    inputs = tokenizer(sen, return_tensors="pt")
    inputs = {k: v.cuda() for k, v in inputs.items()}
    output = model(**inputs)
    pred = torch.argmax(output.logits, dim=-1)
    print(f"输入：{sen}，预测结果：{pred.item()}")


输入：我不想再购买这个商品了，预测结果：0


In [42]:
from transformers import pipeline

model.config.id2label = {0: "负向", 1: "正向"}
pipe = pipeline("text-classification", model=model, tokenizer=tokenizer, device=0)

Device set to use cuda:0


In [43]:

pipe("我非常喜欢这个商品")

[{'label': '正向', 'score': 0.9427462816238403}]